# Script per la creazione delle cartelle
Creo le cartelle `train/`, `test/` e `val/` contenenti il 70%, 15% e 15% delle coppie prese randomicamente

In [1]:
import os
import random
import shutil

In [4]:
def split_dataset(aligned_dir, output_dir,
                  train_ratio=0.70, val_ratio=0.15, test_ratio=0.15,
                  extension=".tif", seed=42):
    """
    Suddivide casualmente le coppie di immagini presenti in `aligned_dir`
    nelle cartelle train/val/test create in `output_dir`, con le proporzioni
    specificate da train_ratio, val_ratio e test_ratio.
    
    Parametri:
    - aligned_dir : cartella contenente tutte le immagini (coppie).
    - output_dir  : cartella dove verranno create train/, val/, test/.
    - train_ratio : percentuale di immagini per il training (default 70%).
    - val_ratio   : percentuale di immagini per la validation (default 15%).
    - test_ratio  : percentuale di immagini per il test (default 15%).
    - extension   : estensione dei file immagine (es. ".png", ".jpg").
    - seed        : seme per la generazione random (riproducibilità).
    """
    # Imposta il seed per ottenere sempre la stessa suddivisione, se necessario
    random.seed(seed)

    # Crea cartelle di destinazione (train/val/test)
    os.makedirs(os.path.join(output_dir, "train"), exist_ok=True)
    os.makedirs(os.path.join(output_dir, "val"), exist_ok=True)
    os.makedirs(os.path.join(output_dir, "test"), exist_ok=True)

    # 1) Individua i file che terminano con '_label_free' + extension
    all_files = os.listdir(aligned_dir)
    label_free_files = sorted([f for f in all_files if f.endswith("_label_free" + extension)])
    print(f"Totale file _label_free trovati: {len(label_free_files)}")

    # 2) Ricava il "prefisso" comune (es. "00000_08500") per ogni coppia
    #    e verifica che esista il corrispondente file "_stained"
    pairs_prefixes = []
    for lf_file in label_free_files:
        prefix = lf_file.replace("_label_free" + extension, "")
        stained_file = prefix + "_stained" + extension
        if stained_file in all_files:
            pairs_prefixes.append(prefix)
    print(f"Totale coppie trovate: {len(pairs_prefixes)}")
    print(f"Elenco primi 50 prefissi: {pairs_prefixes[:50]}")

    # 3) Mescola casualmente i prefissi per suddividere in train/val/test
    random.shuffle(pairs_prefixes)
    num_total = len(pairs_prefixes)

    train_end = int(num_total * train_ratio)
    val_end   = train_end + int(num_total * val_ratio)
    # test_end non serve esplicitamente: tutto quello che rimane va in test

    train_prefixes = pairs_prefixes[:train_end]
    val_prefixes   = pairs_prefixes[train_end:val_end]
    test_prefixes  = pairs_prefixes[val_end:]

    print(f"Totale coppie trovate: {num_total}")
    print(f" - Train: {len(train_prefixes)}")
    print(f" - Val:   {len(val_prefixes)}")
    print(f" - Test:  {len(test_prefixes)}")

    # 4) Funzione per copiare i file dati prefix e cartella di destinazione
    def copy_pair(prefix, subset_folder):
        lf_name = prefix + "_label_free" + extension
        st_name = prefix + "_stained" + extension

        lf_src = os.path.join(aligned_dir, lf_name)
        st_src = os.path.join(aligned_dir, st_name)

        lf_dst = os.path.join(output_dir, subset_folder, lf_name)
        st_dst = os.path.join(output_dir, subset_folder, st_name)

        shutil.copy2(lf_src, lf_dst)
        shutil.copy2(st_src, st_dst)

    # 5) Copia effettiva delle coppie
    for pfx in train_prefixes:
        copy_pair(pfx, "train")
    for pfx in val_prefixes:
        copy_pair(pfx, "val")
    for pfx in test_prefixes:
        copy_pair(pfx, "test")

    print("Suddivisione completata con successo!")


In [8]:
# ESEMPIO DI UTILIZZO:
if __name__ == "__main__":
    aligned_dir = "../../Materiale/Locale/aligned"       # cartella con tutte le 3000 coppie
    output_dir  = "../../Materiale/Locale/dataset_split" # cartella dove creare train/val/test

    split_dataset(aligned_dir, output_dir,
                  train_ratio=0.70, val_ratio=0.15, test_ratio=0.15,
                  extension=".tif", seed=123)

Totale file _label_free trovati: 3039
Totale coppie trovate: 3039
Elenco primi 50 prefissi: ['00000_09300', '00000_09600', '00000_09900', '00000_10200', '00000_10500', '00000_10800', '00000_11100', '00000_11400', '00000_11700', '00000_12000', '00300_08700', '00300_09000', '00300_09300', '00300_09600', '00300_09900', '00300_10200', '00300_10500', '00300_10800', '00300_11100', '00300_11400', '00300_11700', '00300_12000', '00300_12300', '00600_06300', '00600_06600', '00600_06900', '00600_07200', '00600_07500', '00600_07800', '00600_08100', '00600_08400', '00600_08700', '00600_09000', '00600_09300', '00600_09600', '00600_09900', '00600_10200', '00600_10500', '00600_10800', '00600_11100', '00600_11400', '00600_11700', '00600_12000', '00600_12300', '00600_15000', '00600_15300', '00900_05700', '00900_06000', '00900_06300', '00900_06600']
Totale coppie trovate: 3039
 - Train: 2127
 - Val:   455
 - Test:  457
Suddivisione completata con successo!


In [9]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))


PyTorch version: 2.5.1+cu121
CUDA available: True
Device: NVIDIA GeForce RTX 3060 Ti


In [ ]:
# ===========================================================
# FUNZIONA SOLO SU SCRIPT, NON CON NOTEBOOK
# ===========================================================

import os
import time
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

# -------------------------------------------------------
# Dataset personalizzato
# -------------------------------------------------------
class PairedHistologyDataset(Dataset):
    def __init__(self, folder_path, transform=None):
        self.folder_path = folder_path
        self.transform = transform
        self.pairs = self._get_pairs()

    def _get_pairs(self):
        """Cerca tutti i file che finiscono con '_label_free.tif'
           e costruisce la lista dei prefissi."""
        all_files = os.listdir(self.folder_path)
        prefixes = []
        for f in all_files:
            if f.endswith("_label_free.tif"):
                prefix = f.replace("_label_free.tif", "")
                # Controllo che esista anche '_stained.tif'
                stained_file = prefix + "_stained.tif"
                if stained_file in all_files:
                    prefixes.append(prefix)
        return sorted(prefixes)

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        prefix = self.pairs[idx]
        lf_path = os.path.join(self.folder_path, prefix + "_label_free.tif")
        st_path = os.path.join(self.folder_path, prefix + "_stained.tif")

        # Carico entrambe le immagini
        lf_img = Image.open(lf_path).convert("RGB")
        st_img = Image.open(st_path).convert("RGB")

        # Applico le eventuali trasformazioni
        if self.transform:
            lf_img = self.transform(lf_img)
            st_img = self.transform(st_img)

        return lf_img, st_img

# -------------------------------------------------------
# Trasformazioni
# -------------------------------------------------------
transform = transforms.Compose([
    transforms.Resize((512, 512)),   # semplifica
    transforms.ToTensor(),           # converte in tensor [C,H,W] in [0,1]
])

# -------------------------------------------------------
# Semplice modello di test
# -------------------------------------------------------
class SimpleConvNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 3, kernel_size=3, padding=1),
            nn.Tanh()  # output in [-1, 1]
        )

    def forward(self, x):
        return self.net(x)

def main():
    # -------------------------------------------------------
    # Creazione dataset e dataloader
    # -------------------------------------------------------
    train_folder = "../../Materiale/Locale/dataset_split/train"  # <-- METTI il tuo path
    train_dataset = PairedHistologyDataset(train_folder, transform=transform)

    # Imposta batch_size e num_workers=0 per evitare errori su Windows
    train_loader = DataLoader(
        train_dataset,
        batch_size=16,
        shuffle=True,
        num_workers=1,   # 0 => nessun worker parallelo (profiling più leggibile)
        pin_memory=True if torch.cuda.is_available() else False
    )

    # -------------------------------------------------------
    # Inizializzazione
    # -------------------------------------------------------
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device in uso:", device)

    model = SimpleConvNet().to(device)
    criterion = nn.L1Loss()
    optimizer = optim.Adam(model.parameters(), lr=1e-4)

    # -------------------------------------------------------
    # Training loop + mini-profiler con time.time()
    # -------------------------------------------------------
    print("Inizio training di prova...\n")
    model.train()

    num_epochs = 2  # due epoche di test
    for epoch in range(num_epochs):
        running_loss = 0.0
        start_epoch = time.time()

        for i, (input_img, target_img) in enumerate(train_loader):
            # Misura tempo di caricamento + transfer su GPU
            t0 = time.time()
            input_img = input_img.to(device, non_blocking=True)
            target_img = target_img.to(device, non_blocking=True)
            t1 = time.time()

            # Forward + backward pass
            output = model(input_img)
            loss = criterion(output, target_img)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            t2 = time.time()

            running_loss += loss.item()

            # Stampa ogni 10 batch i tempi
            if i % 10 == 0:
                print(f"[Ep {epoch+1:02d} | Batch {i:03d}] "
                    f"Load+Transfer: {(t1 - t0)*1e3:.1f} ms, "
                    f"Fwd+Bwd: {(t2 - t1)*1e3:.1f} ms, "
                    f"Loss: {loss.item():.4f}")

        end_epoch = time.time()
        epoch_time = end_epoch - start_epoch
        avg_loss = running_loss / len(train_loader)
        print(f"Epoch {epoch+1}/{num_epochs} - Loss media: {avg_loss:.4f}, "
            f"Tempo epoch: {epoch_time:.2f} s\n")

    print("Training completato! ✅")

if __name__ == "__main__":
    main()


Device in uso: cuda
Inizio training di prova...



RuntimeError: DataLoader worker (pid(s) 12092) exited unexpectedly